# Evaluate an existing problem and candidate

Use OMMX to check whether a candidate satisfies a problem's conditions and to compute its objective value. **`Instance.evaluate()` checks a supplied candidate; it does not search for a better one.**

We use [QPLIB problem 0018](https://qplib.zib.de/QPLIB_0018.html) and the candidate published on the same page. You do not need to build a model. Knowledge of Python and ordinary variables, objectives, and constraints is enough; no other tutorial's execution state is required.

## 1. Obtain the problem and candidate files

Install Python SDK `3.0.0b7` using steps 1–2 of the [installation guide](../getting_started/install.md). Select that environment as your notebook kernel. No Adapter or solver is needed.

Download the problem's `.qplib` file and the candidate's `.sol` file from the official site. The first download requires an internet connection. This cell saves them under `qplib-example` in the working directory and does not download files that already exist.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("qplib-example")
data_dir.mkdir(exist_ok=True)
for folder, extension in [("qplib", "qplib"), ("sol", "sol")]:
    filename = f"QPLIB_0018.{extension}"
    path = data_dir / filename
    if not path.exists():
        _ = urlretrieve(f"https://qplib.zib.de/{folder}/{filename}", path)

Andrea Scozzari contributed this problem to QPLIB, which is published under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Even if you placed the two files in `qplib-example` yourself, run this cell to define their directory. Existing files do not require a network connection.

## 2. Load and evaluate the candidate

An `Instance` contains the problem's variables, objective, and constraints. A `State` assigns values to variable IDs. Loading a candidate does not establish feasibility. Evaluating the State against the Instance produces a `Solution` with an objective value and feasibility.


In [2]:
from ommx import Instance, State

instance = Instance.load_qplib(str(data_dir / "QPLIB_0018.qplib"))
state = State.load_qplib_solution(
    str(data_dir / "QPLIB_0018.sol"),
    num_variables=len(instance.decision_variables),
)
solution = instance.evaluate(state, atol=1e-8)

print(f"{solution.objective=}")
print(f"{solution.feasible=}")

solution.objective=-6.386014981598351
solution.feasible=True


`objective` is computed by substituting the candidate into the problem's objective, not by accepting the objective value written in the candidate file. This is a minimization problem, so smaller values are better when comparing feasible candidates. `feasible=True` means the variable domains and constraints hold at the specified tolerance. Here `atol=1e-8` allows small floating-point discrepancies.

Pass the original QPLIB problem's number of variables as `num_variables`. This `.sol` omits zero-valued variables; the argument allows the loader to produce the assignment for all 50 variables. Always use problem and candidate files belonging to the same problem.

## 3. Read variable and constraint evaluations

The variables in this problem are nonnegative continuous variables. The following table displays only nonzero values. `id` is the OMMX variable ID, `Continuous` is the variable kind, `lower` and `upper` are bounds, and `value` is the candidate's value. `inf` means there is no finite upper bound.


In [3]:
variables = solution.decision_variables_df()
variables.loc[variables["value"] != 0, ["kind", "lower", "upper", "value"]]

,kind,lower,upper,value
id,,,,
13,Continuous,0.0,inf,0.209637
16,Continuous,0.0,inf,0.275231
38,Continuous,0.0,inf,0.226998
40,Continuous,0.0,inf,0.288135


Next inspect each constraint. The problem requires the variables to sum to 1. After loading, this is represented by two inequalities: the sum must be at most 1 and at least 1.


In [4]:
columns = ["name", "equality", "value", "violation", "feasible"]
solution.constraints_df()[columns]

,name,equality,value,violation,feasible
regular_constraint_id,,,,,
0,Qplib_constr_0 [c_u],<=0,-5.551115e-17,0.000000e+00,True
1,Qplib_constr_0 [c_l],<=0,5.551115e-17,5.551115e-17,True


The leftmost `regular_constraint_id` identifies the loaded constraint. In `name`, `c_u` denotes the upper-side condition and `c_l` the lower-side condition.

- A row with `equality` equal to `<=0` requires its evaluated `value` to be nonpositive.
- `violation` measures the violation. A negative `value` satisfies this inequality and is not a violation.
- `feasible` includes the tolerance. Even a small positive `violation`, such as about `5.55e-17`, can be `True` at this tolerance.

This constraint table alone does not check variable bounds and kinds. Use `solution.feasible` for the candidate as a whole.

## 4. Inspect an infeasible candidate

Evaluate another candidate that assigns zero to every variable. Obtain IDs from the loaded Instance and explicitly provide every value.


In [5]:
zero_state = State(
    {variable.id: 0.0 for variable in instance.decision_variables}
)
zero_solution = instance.evaluate(zero_state, atol=1e-8)
print(f"{zero_solution.objective=}")
print(f"{zero_solution.feasible=}")

constraints = zero_solution.constraints_df()
constraints.loc[~constraints["feasible"], columns]

zero_solution.objective=0.0
zero_solution.feasible=False


,name,equality,value,violation,feasible
regular_constraint_id,,,,,
1,Qplib_constr_0 [c_l],<=0,1.0,1.0,False


The objective is 0, but the candidate violates the lower-side condition requiring the sum to be at least 1. Rows with `feasible=False` identify failed conditions. Filter by this result rather than by a tiny positive `violation`.

You have now loaded an existing problem and candidate and inspected its objective, feasibility, and constraint evaluations. This workflow does not search for candidates and does not prove that the published candidate is globally optimal.

## Related explanations

- Obtain other problems or formats: [Load an existing problem](../how_to/load_existing_instance.md).
- Understand evaluation and numerical rules: [Evaluate a State into a Solution](../user_guide/solution.md).
- Handle several candidates: [SampleSet](../user_guide/sample_set.md).
- Exact candidate-file loading rules: {meth}`~ommx.State.load_qplib_solution`.
